# RL - Chapter 4 - Dynamic Programming

Reinforcement Learning: An Introduction


In [22]:
import numpy as np
import matplotlib.pyplot as plt

# set decimal precision for numpy arrays
np.set_printoptions(precision=1, suppress=True)

## Policy Evaluation


In [23]:
# define Environment
class Environment:
    def __init__(self, states, n_states, goal_states):
        self.states = states
        self.n_s = n_states
        self.goal_states = goal_states
        self.P: dict = None                   # state transition probabilities p(s'|s,a)
        # P = {state: {action: [(probability, next_state, reward, done), ...]}}

In [24]:
# define Policy
class Policy:
    def __init__(self, env: Environment):
        self.env = env
        # for each state, define actions and their probabilities
        self.p_actions = {} # {state: [(action, probability), ...]}
        
        self.initialize_random_policy()
        
    def initialize_random_policy(self):
        for s , a_vals in self.env.P.items():
            actions = list(a_vals.keys())
            n_actions = len(actions)
            self.p_actions[s] = [(a, 1/n_actions) for a in actions] 
    
    def get_action_and_probabilities(self, state):
        return self.p_actions.get(state, [])
    
    def get_probabilities(self, state):
        return [p for a, p in self.get_action_and_probabilities(state)]

In [25]:
# Dynamic Programming Agent
class DP_Agent:
    def __init__(self, env: Environment):
        
        # initialize environment(dynamics) and policy
        self.env = env              # Environment
        self.policy = Policy(env)   # Policy
        
        # setting hyperparameters
        self.gamma = 1.0            # Discount factor
        self.theta = 1e-3           # Convergence threshold
        
        # initialize value function
        self.V = np.zeros(env.n_s)
        
        # counters for diagnostics
        self.n_eval_sweeps = 0
        self.n_improvement_steps = 0
        
    def policy_iteration(self):
        while True:
            self.policy_evaluation()
            policy_stable = self.policy_improvement()
            self.n_improvement_steps += 1
            if policy_stable:
                break

    def policy_evaluation(self):
        while True:
            delta = 0.0
            for s in self.env.states:
                if s in self.env.goal_states:
                    continue

                v_old = self.V[s]
                v_new = 0.0

                for a, action_prob in enumerate(self.policy.get_probabilities(s)):
                    if action_prob == 0.0:
                        continue
                    for prob, next_state, reward, done in self.env.P[s][a]:
                        v_new += action_prob * prob * (reward + self.gamma * self.V[next_state])

                self.V[s] = v_new
                delta = max(delta, abs(v_old - v_new))

            self.n_eval_sweeps += 1
            if delta < self.theta:
                break
            
    def policy_improvement(self):
        ''' Improves the current policy by making it greedy with respect to the current value function V. '''
        policy_stable = True
        threshold = 1e-4

        for s in self.env.states:
            if s in self.env.goal_states:
                continue

            old_action_probabilities = self.policy.get_probabilities(s)
            new_action_probabilities = self.get_greedy_action_probabilities(s)

            # update policy
            self.policy.p_actions[s] = [(a, new_action_probabilities[a]) for a in range(len(new_action_probabilities))]

            if not np.allclose(old_action_probabilities, new_action_probabilities, atol=threshold):
                policy_stable = False

        return policy_stable

    def get_greedy_action_probabilities(self, state):
        ''' Returns the greedy policy for a given state based on the current value function V. '''

        # action_values: q(s,a) for all actions a in state s
        action_values = np.zeros(4)
        for a in range(4):
            for prob, next_state, reward, done in self.env.P[state][a]:
                action_values[a] += prob * (reward + self.gamma * self.V[next_state])
        best_action_value = np.max(action_values)
        n_best_actions = np.sum(action_values == best_action_value)
        p_actions = np.zeros_like(action_values)
        p_actions[action_values == best_action_value] = 1.0 / n_best_actions
        return p_actions

### Grid example


In [26]:
class GridWorldEnv(Environment):
    def __init__(self):
        ''' 4x4 grid world environment '''
        
        # define grid world parameters
        self.n_rows = 4
        self.n_cols = 4
        n_states = self.n_rows * self.n_cols
        states = np.arange(n_states)
        goal_states = [0, n_states - 1]        # goal states
        
        super().__init__(states=states, n_states=n_states, goal_states=goal_states)
        
        # 
        self.n_s = n_states               # total number of states
        self.P = {}                       # transition probabilities p(s'|s,a)
        # P = {state: {action: [(probability, next_state, reward, done), ...]}}
        
        # # environment dynamics:
        # all transitions have -1 reward, except for the goal state which has 0 reward
        # 4 actions: up, down, left, right
        # all actions have same probability of 1/4
        # in the case of hitting the wall, the agent stays in the same state
        
        self.build_transition_probabilities()
    
    def plot_grid(self):
        ''' plot grid with index of each state '''
        grid = np.arange(self.n_s).reshape(self.n_rows, self.n_cols)
        plt.imshow(grid, cmap='viridis')
        for i in range(self.n_rows):
            for j in range(self.n_cols):
                plt.text(j, i, str(grid[i, j]), ha='center', va='center', color='white')
        # plot goal states
        for goal in self.goal_states:
            row, col = divmod(goal, self.n_cols)
            plt.text(col, row-0.2, 'G', ha='center', va='center', color='red', fontsize=12)
        plt.title('Grid World Environment')
        plt.axis('off')
        plt.show()
    
    def build_transition_probabilities(self):
        ''''''
        
        # each P: state -> action -> list of (probability, next_state, reward, done)
        # probability: p(s'|s,a)
        self.P = {s: {a: [] for a in range(4)} for s in range(self.n_s)}
        
        for s in range(self.n_s):
            row, col = divmod(s, self.n_cols)
            if s in self.goal_states:
                continue
            for a in range(4):
                if a == 0:  # up
                    next_row, next_col = max(row - 1, 0), col
                elif a == 1:  # down
                    next_row, next_col = min(row + 1, self.n_rows - 1), col
                elif a == 2:  # left
                    next_row, next_col = row, max(col - 1, 0)
                else:  # right
                    next_row, next_col = row, min(col + 1, self.n_cols - 1)
                
                next_state = next_row * self.n_cols + next_col
                reward = -1 # if next_state not in self.goal_states else 0
                done = (next_state in self.goal_states)
                self.P[s][a].append((1, next_state, reward, done))

In [27]:
# env = GridWorldEnv()
env = GridWorldEnv()
# env.plot_grid()

# run policy evaluation for Grid World environment
agent = DP_Agent(env)

# evaluate the policy
agent.policy_evaluation()

# print state-value function
print("State-value function under the random policy:")
print(agent.V.reshape(env.n_rows, env.n_cols))
print(f"Evaluation sweeps: {agent.n_eval_sweeps}")

State-value function under the random policy:
[[  0. -14. -20. -22.]
 [-14. -18. -20. -20.]
 [-20. -20. -18. -14.]
 [-22. -20. -14.   0.]]
Evaluation sweeps: 88


In [28]:
# policy iteration
agent.policy_iteration()

# print state-value function
print("State-value function after policy iteration:")
print(agent.V.reshape(env.n_rows, env.n_cols))
print(f"Policy improvement steps: {agent.n_improvement_steps}")
print(f"Total evaluation sweeps: {agent.n_eval_sweeps}")

State-value function after policy iteration:
[[ 0. -1. -2. -3.]
 [-1. -2. -3. -2.]
 [-2. -3. -2. -1.]
 [-3. -2. -1.  0.]]
Policy improvement steps: 3
Total evaluation sweeps: 93
